# 03 - Feature Engineering

This notebook covers preparing the cleaned data for modeling: separating features from the target, splitting into train/test sets, and scaling numerical features using standardization (z-score scaling).  

- The **training set** is for fitting the model
- The **test set** will be used to evaluate model performance on unseen data.
- We will use split ratio of **80:20** (80% training and 20% testing).
- Scalling ensures that features with different units and ranges (e.g., `Glucose`, `Age`, `BMI`) are on a comparable scale.

Logic lives in `src/features/feature_engineering.py`.

In [1]:
import sys, os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))
# print("Working directory:", Path.cwd())

In [2]:
from src.data.load_data import load_config, load_data
from src.data.preprocess import clean_data
from src.features.feature_engineering import (
    split_features_target,
    split_train_test,
    scale_features,
)

## Load and clean data

In [3]:
config = load_config()
df = load_data(config)
df_clean = clean_data(df, config["data"]["invalid_zero_columns"])
df_clean.shape

(768, 9)

## Separate features and target

`Outcome` is the target column; the remaining 8 columns are the features fed into the model.

In [4]:
X, y = split_features_target(df_clean, config["data"]["target_column"])
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (768, 8)
y shape: (768,)


## Train/test split

An 80/20 stratified split, `random_state=42`, all driven by `config.yaml`. Stratifying on `y` keeps the class balance consistent between the train and test sets - important given the class imbalance observed in `02_exploratory_data_analysis.ipynb`.

In [5]:
X_train, X_test, y_train, y_test = split_train_test(X, y, config)
print("X_train:", X_train.shape, " X_test:", X_test.shape)

X_train: (614, 8)  X_test: (154, 8)


In [6]:
# Confirm stratification preserved the class balance in both splits
print("y_train class balance:")
print(y_train.value_counts(normalize=True).round(3))
print()
print("y_test class balance:")
print(y_test.value_counts(normalize=True).round(3))

y_train class balance:
Outcome
0    0.651
1    0.349
Name: proportion, dtype: float64

y_test class balance:
Outcome
0    0.649
1    0.351
Name: proportion, dtype: float64


## Feature scaling

`StandardScaler` is fit on the training set only, then applied to both train and test sets. Fitting on train only avoids leaking information about the test set's distribution into the scaling parameters.

In [7]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
print("Scaled X_train mean (should be ~0):", X_train_scaled.mean(axis=0).round(3))
print("Scaled X_train std (should be ~1): ", X_train_scaled.std(axis=0).round(3))

Scaled X_train mean (should be ~0): [-0. -0.  0. -0. -0. -0. -0. -0.]
Scaled X_train std (should be ~1):  [1. 1. 1. 1. 1. 1. 1. 1.]


**Observation:** The test set's scaled mean/std aren't expected to be exactly 0/1 - the scaler's parameters come from the training set only, so applying them to test data will naturally shift slightly. Seeing them close to 0/1 just reflects that train and test are similarly distributed here, because of the stratified split.

In [8]:
print("Scaled X_test mean:", X_test_scaled.mean(axis=0).round(3))
print("Scaled X_test std: ", X_test_scaled.std(axis=0).round(3))

Scaled X_test mean: [ 0.039 -0.002  0.1    0.037  0.188  0.006 -0.084 -0.053]
Scaled X_test std:  [1.081 1.071 0.922 0.942 1.409 1.038 1.013 0.968]


## Next steps

With train/test data split and scaled, `04_modeling.ipynb` covers training the baseline and logistic regression models, evaluating them, and interpreting the results.